# Time-Series Forecasting with ARIMA / SARIMA and Prophet

End-to-end pipeline on **daily demand** data:

1. Load data and explore trend / seasonality
2. Decompose into **Trend + Seasonality + Residual**
3. **ADF test** for stationarity and apply differencing
4. Train **SARIMA** (grid search on AIC) and **Prophet** (hyperparameter search on a validation window)
5. Evaluate on a 30-day **holdout test split** using **MAPE** and **RMSE** (plus a seasonal-naive baseline)
6. Refit the best model on all data and produce a **30-day future forecast**

> **Data note:** The dataset file linked in the project was not available, so this notebook generates a *synthetic* daily demand series
> (trend + weekly + yearly seasonality + noise + promo spikes) and saves it as `daily_demand.csv`.
> To use your real file, put a CSV with columns `date,demand` in the same folder named `daily_demand.csv` and re-run - the generator cell is skipped automatically.

In [ ]:
# Run once if packages are missing
# !pip install -q statsmodels prophet scikit-learn matplotlib pandas numpy

In [ ]:
import os, warnings, logging, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from prophet import Prophet

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

plt.rcParams.update({"figure.figsize": (13, 4), "axes.grid": True, "grid.alpha": 0.3})
SEED = 42
HORIZON = 30        # test split length AND future forecast length
DATA_PATH = "daily_demand.csv"

## 1. Load / generate the data

In [ ]:
if not os.path.exists(DATA_PATH):
    rng = np.random.default_rng(SEED)
    n = 1095                                   # 3 years of daily data
    dates = pd.date_range("2022-01-01", periods=n, freq="D")
    t = np.arange(n)
    doy = dates.dayofyear.values

    trend   = 200 + 0.18 * t                                         # steady growth
    weekly  = np.array([-6, -4, 0, 5, 14, 32, 20])[dates.dayofweek]  # Mon..Sun, weekend peak
    yearly  = 25 * np.sin(2 * np.pi * (doy - 80) / 365.25) + 8 * np.sin(4 * np.pi * doy / 365.25)

    noise = np.zeros(n)                                              # AR(1) noise
    eps = rng.normal(0, 6, n)
    for i in range(1, n):
        noise[i] = 0.5 * noise[i - 1] + eps[i]

    promo = np.zeros(n)
    promo[rng.choice(n, 12, replace=False)] = 40                     # occasional promo spikes

    demand = np.clip(trend + weekly + yearly + noise + promo, 0, None).round(2)
    pd.DataFrame({"date": dates, "demand": demand}).to_csv(DATA_PATH, index=False)
    print("Synthetic dataset generated ->", DATA_PATH)
else:
    print("Found existing", DATA_PATH, "- using it.")

df = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date")
series = df.set_index("date")["demand"].asfreq("D")
series = series.interpolate()                 # safety net for gaps in a real file
print(series.shape, series.index.min().date(), "->", series.index.max().date())
print("Missing values:", int(series.isna().sum()))
series.describe().round(2)

In [ ]:
fig, ax = plt.subplots()
series.plot(ax=ax, lw=0.9)
ax.axvline(series.index[-HORIZON], color="red", ls="--", label="Holdout starts")
ax.set_title("Daily demand - full history"); ax.set_ylabel("Demand"); ax.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
series.groupby(series.index.dayofweek).mean().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], rotation=0)
axes[0].set_title("Average demand by day of week")
series.groupby(series.index.month).mean().plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Average demand by month"); axes[1].set_xlabel("Month")
plt.tight_layout(); plt.show()

## 2. Train / test split

The last **30 days** are held out as the test set. All model selection below uses **training data only**, so the test score is an honest estimate.

In [ ]:
train, test = series.iloc[:-HORIZON], series.iloc[-HORIZON:]
print(f"Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} days)")
print(f"Test : {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} days)")

def rmse(y, yhat): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat)) ** 2)))
def mape(y, yhat): return float(np.mean(np.abs((np.asarray(y) - np.asarray(yhat)) / np.asarray(y))) * 100)

## 3. Decomposition: Trend + Seasonality + Residual

STL (robust to outliers such as promo spikes) is run twice: once for the **weekly** cycle (period 7) and once for the **yearly** cycle (period 365).

In [ ]:
def plot_decomp(res, title):
    fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
    for ax, (name, comp) in zip(axes, [("Observed", res.observed), ("Trend", res.trend),
                                       ("Seasonal", res.seasonal), ("Residual", res.resid)]):
        ax.plot(comp, lw=0.8); ax.set_ylabel(name)
    axes[0].set_title(title)
    plt.tight_layout(); plt.show()

stl_weekly = STL(train, period=7, robust=True).fit()
plot_decomp(stl_weekly, "STL decomposition - weekly seasonality (period = 7)")

stl_yearly = STL(train, period=365, robust=True).fit()
plot_decomp(stl_yearly, "STL decomposition - yearly seasonality (period = 365)")

strength_trend = max(0, 1 - stl_weekly.resid.var() / (stl_weekly.trend + stl_weekly.resid).var())
strength_seas  = max(0, 1 - stl_weekly.resid.var() / (stl_weekly.seasonal + stl_weekly.resid).var())
print(f"Trend strength (0-1)          : {strength_trend:.2f}")
print(f"Weekly seasonality strength   : {strength_seas:.2f}")

## 4. Stationarity - Augmented Dickey-Fuller (ADF) test

* H0: the series has a unit root (**non-stationary**)
* If **p < 0.05** -> reject H0 -> series is stationary

We test the raw series, then apply **first differencing** (removes trend, d=1) and **seasonal differencing** (lag 7, D=1).

In [ ]:
def adf_report(x, name):
    x = pd.Series(x).dropna()
    stat, p, usedlag, nobs, crit, _ = adfuller(x, autolag="AIC")
    verdict = "STATIONARY" if p < 0.05 else "NON-stationary"
    print(f"{name:<38} ADF = {stat:8.3f} | p-value = {p:.4f} | lags = {usedlag:<2} -> {verdict}")
    return p

p_raw  = adf_report(train, "Original series")
d1     = train.diff()
p_d1   = adf_report(d1, "First difference (d=1)")
d1s7   = train.diff().diff(7)
p_d1s7 = adf_report(d1s7, "First + seasonal diff (d=1, D=1, m=7)")

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
for ax, (s, name) in zip(axes, [(train, "Original"), (d1, "d = 1"), (d1s7, "d = 1, D = 1 (lag 7)")]):
    ax.plot(s, lw=0.7); ax.set_ylabel(name)
axes[0].set_title("Effect of differencing")
plt.tight_layout(); plt.show()

# differencing order chosen from the ADF results
d = 0 if p_raw < 0.05 else 1
print("\nChosen non-seasonal differencing order d =", d)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(d1s7.dropna(),  lags=40, ax=axes[0], title="ACF  (d=1, D=1, m=7)")
plot_pacf(d1s7.dropna(), lags=40, ax=axes[1], title="PACF (d=1, D=1, m=7)", method="ywm")
plt.tight_layout(); plt.show()

## 5. Baseline - Seasonal Naive

Repeat the last observed week. Any real model should beat this.

In [ ]:
results = {}
snaive_pred = np.tile(train.iloc[-7:].values, int(np.ceil(HORIZON / 7)))[:HORIZON]
results["Seasonal Naive"] = {"MAPE (%)": mape(test, snaive_pred), "RMSE": rmse(test, snaive_pred)}
preds = {"Seasonal Naive": pd.Series(snaive_pred, index=test.index)}
pd.DataFrame(results).T.round(3)

## 6. SARIMA - hyperparameter selection

Grid search over `(p, d, q)(P, D, Q, 7)`, picking the lowest **AIC** on the training data.

In [ ]:
m = 7
p_rng, q_rng = range(0, 3), range(0, 3)
P_rng, Q_rng = range(0, 2), range(0, 2)
D_rng = [0, 1]

rows = []
for p, q, P, D, Q in itertools.product(p_rng, q_rng, P_rng, D_rng, Q_rng):
    try:
        fit = SARIMAX(train, order=(p, d, q), seasonal_order=(P, D, Q, m),
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        rows.append({"order": (p, d, q), "seasonal_order": (P, D, Q, m), "AIC": fit.aic, "BIC": fit.bic})
    except Exception as e:
        continue

grid = pd.DataFrame(rows).sort_values("AIC").reset_index(drop=True)
print(f"{len(grid)} SARIMA models fitted. Top 5 by AIC:")
grid.head(5)

In [ ]:
best_order, best_sorder = grid.loc[0, "order"], grid.loc[0, "seasonal_order"]
print("Best SARIMA:", best_order, best_sorder)

sarima_fit = SARIMAX(train, order=best_order, seasonal_order=best_sorder,
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarima_fc = sarima_fit.get_forecast(steps=HORIZON)
sarima_pred = pd.Series(sarima_fc.predicted_mean.values, index=test.index)
sarima_ci = sarima_fc.conf_int(alpha=0.05); sarima_ci.index = test.index

preds["SARIMA"] = sarima_pred
results["SARIMA"] = {"MAPE (%)": mape(test, sarima_pred), "RMSE": rmse(test, sarima_pred)}
print(f"SARIMA  ->  MAPE = {results['SARIMA']['MAPE (%)']:.2f}%   RMSE = {results['SARIMA']['RMSE']:.2f}")

In [ ]:
# Residual diagnostics for the SARIMA model
sarima_fit.plot_diagnostics(figsize=(13, 8)); plt.tight_layout(); plt.show()
lb = acorr_ljungbox(sarima_fit.resid.iloc[max(best_sorder[3], 1) * m:], lags=[7, 14, 21], return_df=True)
print("Ljung-Box test (p > 0.05 means residuals look like white noise):")
lb.round(4)

## 7. Prophet - hyperparameter selection

Prophet is tuned on a **validation window** (last 30 days of the training set), then re-evaluated on the real test set.

In [ ]:
def prophet_frame(s): return pd.DataFrame({"ds": s.index, "y": s.values})

def make_prophet(**kw):
    return Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, **kw)

inner_train, val = train.iloc[:-HORIZON], train.iloc[-HORIZON:]
param_grid = {
    "changepoint_prior_scale": [0.01, 0.05, 0.1, 0.5],
    "seasonality_mode": ["additive", "multiplicative"],
}

rows = []
for cps, mode in itertools.product(*param_grid.values()):
    mdl = make_prophet(changepoint_prior_scale=cps, seasonality_mode=mode)
    mdl.fit(prophet_frame(inner_train))
    fc = mdl.predict(mdl.make_future_dataframe(periods=HORIZON))
    yhat = fc["yhat"].iloc[-HORIZON:].values
    rows.append({"changepoint_prior_scale": cps, "seasonality_mode": mode,
                 "val_RMSE": rmse(val, yhat), "val_MAPE (%)": mape(val, yhat)})

prophet_grid = pd.DataFrame(rows).sort_values("val_RMSE").reset_index(drop=True)
prophet_grid

In [ ]:
best_p = prophet_grid.iloc[0]
best_params = {"changepoint_prior_scale": float(best_p["changepoint_prior_scale"]),
               "seasonality_mode": best_p["seasonality_mode"]}
print("Best Prophet params:", best_params)

prophet_fit = make_prophet(**best_params)
prophet_fit.fit(prophet_frame(train))
p_fc = prophet_fit.predict(prophet_fit.make_future_dataframe(periods=HORIZON))
p_test = p_fc.iloc[-HORIZON:]
prophet_pred = pd.Series(p_test["yhat"].values, index=test.index)

preds["Prophet"] = prophet_pred
results["Prophet"] = {"MAPE (%)": mape(test, prophet_pred), "RMSE": rmse(test, prophet_pred)}
print(f"Prophet ->  MAPE = {results['Prophet']['MAPE (%)']:.2f}%   RMSE = {results['Prophet']['RMSE']:.2f}")

fig = prophet_fit.plot_components(p_fc); plt.show()

## 8. Model comparison on the holdout test split

In [ ]:
comparison = pd.DataFrame(results).T.sort_values("RMSE").round(3)
display(comparison)

fig, ax = plt.subplots(figsize=(13, 5))
train.iloc[-90:].plot(ax=ax, color="grey", label="Train (last 90 days)")
test.plot(ax=ax, color="black", lw=2, label="Actual (test)")
preds["SARIMA"].plot(ax=ax, label=f"SARIMA  (MAPE {results['SARIMA']['MAPE (%)']:.1f}%)")
preds["Prophet"].plot(ax=ax, label=f"Prophet (MAPE {results['Prophet']['MAPE (%)']:.1f}%)")
preds["Seasonal Naive"].plot(ax=ax, ls=":", label="Seasonal Naive")
ax.fill_between(test.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], alpha=0.12, label="SARIMA 95% CI")
ax.set_title("30-day holdout: forecast vs actual"); ax.legend(ncol=3); plt.show()

best_model = comparison.index[comparison.index != "Seasonal Naive"][0]
print("Best model on test split:", best_model)

## 9. Final model - refit on ALL data and forecast the next 30 days

The winning model is retrained on the full history (train + test) so the forecast uses the most recent information.

In [ ]:
future_idx = pd.date_range(series.index[-1] + pd.Timedelta(days=1), periods=HORIZON, freq="D")

# SARIMA refit on full data
sarima_full = SARIMAX(series, order=best_order, seasonal_order=best_sorder,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sf = sarima_full.get_forecast(steps=HORIZON)
sarima_future = pd.DataFrame({"forecast": sf.predicted_mean.values}, index=future_idx)
ci = sf.conf_int(alpha=0.05); sarima_future["lower"], sarima_future["upper"] = ci.iloc[:, 0].values, ci.iloc[:, 1].values

# Prophet refit on full data
prophet_full = make_prophet(**best_params).fit(prophet_frame(series))
pf = prophet_full.predict(prophet_full.make_future_dataframe(periods=HORIZON)).iloc[-HORIZON:]
prophet_future = pd.DataFrame({"forecast": pf["yhat"].values, "lower": pf["yhat_lower"].values,
                               "upper": pf["yhat_upper"].values}, index=future_idx)

final = sarima_future if best_model == "SARIMA" else prophet_future

fig, ax = plt.subplots(figsize=(13, 5))
series.iloc[-120:].plot(ax=ax, color="grey", label="History (last 120 days)")
final["forecast"].plot(ax=ax, color="crimson", lw=2, label=f"{best_model} 30-day forecast")
ax.fill_between(final.index, final["lower"], final["upper"], color="crimson", alpha=0.15, label="95% interval")
other = prophet_future if best_model == "SARIMA" else sarima_future
other["forecast"].plot(ax=ax, ls="--", color="steelblue", label="Other model (for reference)")
ax.set_title(f"30-day future demand forecast: {future_idx[0].date()} -> {future_idx[-1].date()}")
ax.set_ylabel("Demand"); ax.legend(); plt.show()

out = final.round(2).rename_axis("date").reset_index()
out.to_csv("forecast_30_days.csv", index=False)
out.head(10)

## 10. Summary

* **Decomposition** shows an upward trend, a strong weekly cycle (weekend peak), and a smooth yearly cycle.
* **ADF test**: the raw series is non-stationary (trend); first differencing (d = 1) makes it stationary, and seasonal differencing at lag 7 removes the weekly cycle.
* **SARIMA** orders were chosen by AIC grid search; **Prophet** by validation RMSE over changepoint flexibility and seasonality mode - both using training data only.
* Models are compared with **MAPE** and **RMSE** on the 30-day holdout and against a seasonal-naive baseline.
* The best model is refit on all data and the **30-day forecast** (with 95% interval) is saved to `forecast_30_days.csv`.

**Possible extensions:** add holiday/promo regressors (SARIMAX exogenous / Prophet `add_regressor`), rolling-origin cross-validation, and ensembling SARIMA + Prophet.